# 02_内容构建 (Content Builder Agent)

**来源:** [LangChain Docs — Build a content builder agent](https://docs.langchain.com/deep-agents/content-builder)

本教程演示如何使用 Deep Agents 构建一个内容写作智能体。

## 1. 核心概念

| 概念 | 说明 |
|------|------|
| 长期记忆 (Memory) | 通过 AGENTS.md 文件定义品牌语调和写作标准 |
| 技能 (Skills) | skills/ 目录中的 SKILL.md 文件，定义特定内容类型的写作流程 |
| 子智能体 | 用于搜索研究的专用子智能体 |
| 文件系统后端 | 用于读写帖子、研究笔记和图片 |
| 自定义工具 | Web 搜索和 AI 图片生成 |

## 2. 环境准备

安装依赖并设置 API 密钥。

In [ ]:
# 安装核心依赖
# pip install deepagents google-genai pillow pyyaml rich tavily-python langchain

import os
from dotenv import load_dotenv
load_dotenv()
print("环境准备完成")


## 3. 品牌记忆文件 (AGENTS.md)

AGENTS.md 文件定义了品牌的语调、写作标准和内容支柱。当智能体创建时，该文件被加载到系统提示中。

In [ ]:
AGENTS_MD_CONTENT = """# 内容写手智能体

你是一家科技公司的内容写手。

## 品牌语调
- **专业但不拘谨**：像知识渊博的同事一样写作
- **清晰直接**：除非必要，避免行话
- **自信但不傲慢**：分享专业知识，不说教
- **有吸引力**：使用具体例子、类比和故事

## 写作标准
1. **使用主动语态**：智能体处理请求，而不是请求被处理
2. **以价值引领**：从读者关心的内容开始
3. **一段一意**：保持段落聚焦、易浏览
4. **具体而非抽象**：使用具体例子和数字
5. **以行动结尾**：每篇文章让读者知道下一步做什么

## 内容支柱
- AI 智能体和自动化
- 开发者工具和生产力
- 软件架构和最佳实践
- 新兴技术和趋势

## 格式指南
- 使用标题(H2/H3)分割内容
- 在相关处包含代码示例
- 3 项以上的列表使用项目符号
- 尽量保持句子在 25 字以内

## 研究要求
写作前：
1. 使用 researcher 子智能体进行深入主题研究
2. 收集至少 3 个可信来源
3. 确定读者需要理解的关键点
"""

from pathlib import Path
project_dir = Path("content-builder-agent")
project_dir.mkdir(exist_ok=True)
with open(project_dir / "AGENTS.md", "w") as f:
    f.write(AGENTS_MD_CONTENT)
print(f"AGENTS.md 已写入 {project_dir / "AGENTS.md"}")


## 4. 子智能体配置 (subagents.yaml)

定义研究子智能体，配置其使用的模型、工具和系统提示。

In [ ]:
import yaml

SUBAGENTS_CONFIG = {
    "researcher": {
        "description": "研究任何主题时，始终优先使用此子智能体。搜索 Web 获取最新信息。",
        "model": "deepseek-v4-flash",
        "system_prompt": (
            "你是一名研究助理。你可以使用 web_search 和 write_file 工具。\n\n"
            "## 工作流程\n"
            "1. 使用 web_search 查找主题信息\n"
            "2. 进行 2-3 次针对性搜索\n"
            "3. 收集关键统计数据、引用和例子\n"
            "4. 将结果保存到任务指定的文件路径\n\n"
            "## 注意事项\n"
            "- 始终在研究结果中包含来源 URL\n"
            "- 保持结果简洁但有信息量"
        ),
        "tools": ["web_search"]
    }
}

with open(project_dir / "subagents.yaml", "w") as f:
    yaml.dump(SUBAGENTS_CONFIG, f, default_flow_style=False, allow_unicode=True)
print(f"subagents.yaml 已写入")


## 5. 技能文件夹 (Skills)

技能文件夹包含特定内容类型的写作模板。每个技能是一个文件夹中的 SKILL.md 文件，包含 YAML 前置元数据和写作指令。

In [ ]:
skills_blog = project_dir / "skills" / "blog-post"
skills_blog.mkdir(parents=True, exist_ok=True)

SKILL_BLOG_POST = """---
name: blog-post
description: 撰写和编排长文博客、创建教程大纲、优化 SEO 并生成封面图片。
---

# 博客写作技能

## 研究优先（必需）
- 使用 task 工具，subagent_type 设为 "researcher"
- 同时指定主题和保存研究结果的路径
- 研究完成后，先读取结果文件再开始写作

## 输出结构
- blogs/<slug>/post.md - 博客文章正文
- blogs/<slug>/hero.png - 封面图片（必需）

## 博客文章结构
1. 钩子（开头） - 引人入胜的问题或陈述
2. 背景（问题） - 为什么这很重要
3. 主要内容（解决方案） - 3-5 个主要章节
4. 实践应用 - 逐步操作指南
5. 结论与行动号召 - 关键要点和下一步

## SEO 优化
- 主关键词在标题和首段中
- 关键词自然使用 3-5 次
- 标题不超过 60 字
- Meta 描述 150-160 字

## 封面图片生成
- 写作完成后使用 generate_cover 工具
- 提示词应包含：主题、风格、构图、配色、光线
"""

with open(skills_blog / "SKILL.md", "w") as f:
    f.write(SKILL_BLOG_POST)
print(f"博客技能已创建")


In [ ]:
skills_social = project_dir / "skills" / "social-media"
skills_social.mkdir(parents=True, exist_ok=True)

SKILL_SOCIAL_MEDIA = """---
name: social-media
description: 撰写引人入胜的社交媒体帖子、编写钩子、推荐话题标签、创建线程结构并生成配图。
---

# 社交媒体内容技能

## 研究优先（必需）
- 使用 task 工具，subagent_type 设为 "researcher"
- 同时指定主题和研究结果保存路径

## 输出结构
- LinkedIn: linkedin/<slug>/post.md + image.png
- Twitter/X: tweets/<slug>/thread.md + image.png

## LinkedIn 指南
- 1300 字限制
- 首行必须抓住注意力
- 结尾 3-5 个话题标签
- 专业但不失个人风格

## Twitter/X 指南
- 每推 280 字限制
- 线程使用 1/n 格式
- 每推最多 2 个话题标签

## 配图最佳实践
- 大胆简洁的构图
- 高对比度（滚动时突出）
- 单一焦点
- 方形或 4:5 比例

## 内容类型
- 公告贴：以新闻引领，解释影响
- 洞察贴：分享一个心得，给出可操作建议
- 提问贴：提出真实问题，先给出你的看法
"""

with open(skills_social / "SKILL.md", "w") as f:
    f.write(SKILL_SOCIAL_MEDIA)
print(f"社交媒体技能已创建")


## 6. 自定义工具实现

定义 Web 搜索和 AI 图片生成工具，供主智能体和子智能体使用。

In [ ]:
import os
from pathlib import Path
from typing import Literal
from langchain.tools import tool

EXAMPLE_DIR = Path.cwd() / "content-builder-agent"

@tool
def web_search(query: str, max_results: int = 5,
               topic: Literal['general','news'] = 'general') -> dict:
    """搜索 Web 获取最新信息。
    参数:
        query: 搜索查询（请尽量具体和详细）
        max_results: 返回结果数（默认: 5）
    返回:
        包含标题、URL 和内容摘要的搜索结果。
    """
    try:
        from tavily import TavilyClient
        api_key = os.environ.get("TAVILY_API_KEY")
        if not api_key:
            return {"error": "TAVILY_API_KEY 未设置"}
        return TavilyClient(api_key=api_key).search(query, max_results=max_results)
    except Exception as e:
        return {"error": f"搜索失败: {e}"}

@tool
def generate_cover(prompt: str, slug: str) -> str:
    """为博客文章生成封面图片。
    参数:
        prompt: 要生成图片的详细描述
        slug: 博客文章的 slug
    """
    # 集成 Gemini 或其他图片生成 API
    output_path = EXAMPLE_DIR / "blogs" / slug / "hero.png"
    output_path.parent.mkdir(parents=True, exist_ok=True)
    return f"图片已保存至 {output_path}"

@tool
def generate_social_image(prompt: str, platform: str, slug: str) -> str:
    """为社交媒体帖子生成配图。
    参数:
        prompt: 图片描述
        platform: linkedin 或 tweets
        slug: 帖子 slug
    """
    output_path = EXAMPLE_DIR / platform / slug / "image.png"
    output_path.parent.mkdir(parents=True, exist_ok=True)
    return f"图片已保存至 {output_path}"

print("所有自定义工具定义完成")


## 7. 加载子智能体配置

从 YAML 文件中读取子智能体定义，并将工具名称解析为实际的函数对象。

In [ ]:
import yaml
from pathlib import Path

def load_subagents(config_path: Path):
    """从 YAML 加载子智能体定义并绑定工具"""
    available_tools = {'web_search': web_search}
    with open(config_path) as f:
        config = yaml.safe_load(f)
    subagents = []
    for name, spec in config.items():
        sa = {'name': name, 'description': spec['description'],
              'system_prompt': spec['system_prompt']}
        if 'model' in spec:
            sa['model'] = spec['model']
        if 'tools' in spec:
            sa['tools'] = [available_tools[t] for t in spec['tools']]
        subagents.append(sa)
    return subagents

subagents = load_subagents(EXAMPLE_DIR / 'subagents.yaml')
print(f"已加载 {len(subagents)} 个子智能体")


## 8. 创建内容写作智能体

整合所有组件：模型、记忆、技能、工具、子智能体和文件系统后端。

In [ ]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend

def create_content_writer():
    """创建由文件系统配置驱动的内容写作智能体"""
    return create_deep_agent(
        model="deepseek-v4-flash",
        memory=["./AGENTS.md"],
        skills=["./skills/"],
        tools=[generate_cover, generate_social_image],
        subagents=load_subagents(EXAMPLE_DIR / "subagents.yaml"),
        backend=FilesystemBackend(root_dir=EXAMPLE_DIR),
    )

print("内容写作智能体创建函数已定义")


## 9. 运行内容生成任务

向智能体提交内容创作请求，观察它如何研究、撰写并配图。

In [ ]:
# 示例：运行内容撰写任务
# 注意：需要有效的 API 密钥
# agent = create_content_writer()
# result = agent.invoke({
#     "messages": [{"role": "user", "content": "写一篇关于 AI 智能体在企业级软件开发中崛起的博客文章"}]
# })
print("运行内容生成任务（取消注释以执行）")


## 10. 文件产出结构

| 目录 | 内容 |
|------|------|
| blogs/<slug>/post.md | 博客文章正文 |
| blogs/<slug>/hero.png | 博客封面图片 |
| linkedin/<slug>/post.md | LinkedIn 帖子内容 |
| linkedin/<slug>/image.png | LinkedIn 配图 |
| tweets/<slug>/thread.md | Twitter/X 线程内容 |
| tweets/<slug>/image.png | Twitter 配图 |
| research/<slug>.md | 研究笔记 |

---

**参考链接**
- [LangChain Docs — Deep Agents Content Builder](https://docs.langchain.com/deep-agents/content-builder)
- [Tavily Search API](https://tavily.com)
